# Large-ASR zero-shot benchmark (Whisper large-v3 / NVIDIA Canary-1B / Qwen2-Audio-7B-Instruct)

Runs `host_software/ml_audio/evaluations/benchmark_large_asr_models.py` against the same held-out `val` split (`build_manifest_entries(DEFAULT_DATASET_ROOT, "val", labels)`, identical to what `evaluate_nemo_checkpoint.py` uses) that every NeMo MatchboxNet checkpoint in this plan has been scored against.

**Why this runs on Colab/GPU and not the local dev machine:** measured (not estimated) on the local CPU-only machine, Whisper large-v3 averaged 44.4s/clip (`evaluations/reports/asr_benchmark_whisper-large-v3_20260918T080504Z.json`, 5-clip smoke test) -> ~30 hours extrapolated for the full 2448-clip split. Qwen2-Audio-7B-Instruct's fp32 weights alone are ~28GB against that machine's 34GB total RAM -- too tight to run reliably. Both fit comfortably in a Colab GPU runtime.

**What this is actually for:** the user reframed this benchmark's purpose -- it is NOT "large model vs our small model, final accuracy." These are **zero-shot only** (no fine-tuning) reference points, used as fixed ceilings against a separate fine-tuning effort/accuracy learning curve already built for the ~70K-param (74,892) NeMo MatchboxNet classifier from its own checkpoint history (epoch counts, noisemix vs. not, seed0/1/4, etc. -- see `docs/plans/audio_eval_notebook_refactor_plan.md`). Do not fine-tune any of the three models here.

**Before running:** Runtime -> Change runtime type -> GPU. Needs the current data package from `prepare_colab_package.py` (same zip/Drive flow as `colab_nemo_finetune.ipynb` -- deliberately not the superseded DVC/git-clone approach, per the refactor plan's "Colab Data Flow: DVC Revert Was Intentional, 2026-09-17" entry).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU -- Runtime > Change runtime type > GPU, then re-run this cell. "
          "This benchmark is specifically deferred to GPU; running it CPU-only defeats the point.")

In [ ]:
!pip install -q transformers accelerate psutil soundfile
!pip install -q "nemo_toolkit[asr]"   # only needed for --model canary-1b

## Get the code + dataset

Same package format as `colab_nemo_finetune.ipynb` -- rebuild via `prepare_colab_package.py` locally first if the zip on Drive predates recent dataset changes.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Update this to wherever you uploaded the (freshly rebuilt) zip in Drive.
DRIVE_ZIP_PATH = "/content/drive/MyDrive/ml_audio_colab_package.zip"

# Reports land here so they survive after the Colab runtime is recycled --
# same convention as DRIVE_OUTPUT_DIR in colab_nemo_finetune.ipynb.
import os
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ml_audio_asr_benchmark_output"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print("Output directory:", DRIVE_OUTPUT_DIR)

In [ ]:
import zipfile
import sys

os.makedirs("host_software", exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP_PATH) as zf:
    zf.extractall("host_software")

HOST_SOFTWARE_DIR = os.path.abspath("host_software")
if HOST_SOFTWARE_DIR not in sys.path:
    sys.path.insert(0, HOST_SOFTWARE_DIR)

print(os.listdir(os.path.join(HOST_SOFTWARE_DIR, "ml_audio")))

In [ ]:
# Same dataset root every NeMo checkpoint in this plan was scored against --
# reused here so the zero-shot numbers are on the exact same held-out split,
# not a fresh one.
from ml_audio.training.train_audio_command_classifier import DEFAULT_DATASET_ROOT

print("Dataset root:", DEFAULT_DATASET_ROOT)
print("Exists:", os.path.isdir(DEFAULT_DATASET_ROOT))

BENCHMARK_SCRIPT = os.path.join(HOST_SOFTWARE_DIR, "ml_audio", "evaluations", "benchmark_large_asr_models.py")
assert os.path.exists(BENCHMARK_SCRIPT), BENCHMARK_SCRIPT

## Smoke test

Confirms the model loads and the transcribe -> parse-command -> score path runs end-to-end on a handful of clips before committing to a full run. Matches the 5-clip smoke test already done locally for Whisper (44.4s/clip on CPU) -- expect this to be dramatically faster on GPU.

In [ ]:
!python "{BENCHMARK_SCRIPT}" --model whisper-large-v3 --limit 5 --dataset-root "{DEFAULT_DATASET_ROOT}"

## Full runs -- zero-shot only, no fine-tuning

Each of these transcribes the full held-out split, maps the free-text transcript onto our 12-class command vocabulary (`_background_, backward, forward, go_blue, go_green, go_grey, go_red, go_yellow, hold, left, right, stop` -- `go_black` is not a trained class and is deliberately excluded from the keyword map), and scores it the same way the NeMo confusion-matrix reports do, plus latency and footprint. Run one at a time -- these are large models and Colab's GPU memory is shared across whatever else is loaded in this session.

In [ ]:
!python "{BENCHMARK_SCRIPT}" --model whisper-large-v3 --dataset-root "{DEFAULT_DATASET_ROOT}"

In [ ]:
!python "{BENCHMARK_SCRIPT}" --model canary-1b --dataset-root "{DEFAULT_DATASET_ROOT}"

In [ ]:
# Largest of the three (7B params) -- if this OOMs on the assigned Colab GPU
# tier, that itself is a real finding worth recording (footprint is part of
# what this benchmark measures), not just an error to route around.
!python "{BENCHMARK_SCRIPT}" --model qwen2-audio-7b --dataset-root "{DEFAULT_DATASET_ROOT}"

## Copy reports back to Drive

The Colab runtime's local disk does not persist -- without this step the reports are lost when the session ends.

In [ ]:
import glob
import shutil

report_dir = os.path.join(HOST_SOFTWARE_DIR, "ml_audio", "evaluations", "reports")
copied = []
for report_path in glob.glob(os.path.join(report_dir, "asr_benchmark_*.json")):
    dest = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(report_path))
    shutil.copy(report_path, dest)
    copied.append(dest)

print(f"Copied {len(copied)} report(s) to Drive:")
for path in copied:
    print(" ", path)

## Not yet done

- These zero-shot numbers still need to be folded into the fine-tuning effort/accuracy curve for the 70K-param MatchboxNet (built from checkpoint-filename + refactor-plan history) as horizontal reference lines -- that combined table is the actual deliverable, not this notebook's raw per-model JSON reports on their own.
- `qwen2-audio-7b` has not been run anywhere yet (local or Colab) as of this notebook's creation -- its first real numbers, including whether it completes at all on a standard Colab GPU tier, come from running this notebook.
- If `canary-1b`'s NeMo download lands in the default HF cache instead of a project-convention path (as it did locally), that's cosmetic, not a correctness issue -- no need to fix it here.